# Qwen3 ファインチューニング (LoRA / QLoRA)

本ノートブックでは以下の流れでファインチューニングを行います。
1. Google Drive のマウントとリポジトリの clone、依存パッケージのインストール
2. ベースモデル / データセット / 出力先などの設定とランタイム再起動後の復元
3. データセットの読み込みとフォーマット整形
4. LoRA / QLoRA 設定での学習実行
5. 任意で LoRA をベースモデルへマージ

再起動時は「2. 設定セル」を再実行すると保存済みの設定が復旧されます。


In [ ]:
# === 1. Colab utility setup ===
import os
import shlex
import sys
from typing import Sequence

try:
    from IPython import get_ipython
except ImportError:  # pragma: no cover
    get_ipython = None  # type: ignore


def _require_ipython():
    ip = get_ipython() if callable(get_ipython) else None
    if ip is None:
        raise RuntimeError("IPython 上で実行してください。Google Colab を想定しています。")
    return ip


def clone_repo(url: str, target: str, branch: str | None = None) -> None:
    if os.path.exists(target):
        print("既存のリポジトリを再利用します:", target)
        return
    ip = _require_ipython()
    if branch:
        print(f"リポジトリを clone します: {url} (branch={branch})")
        ip.system(f"git clone --branch {branch} --single-branch {url} {target}")
    else:
        print("リポジトリを clone します:", url)
        ip.system(f"git clone {url} {target}")


def _run_pip(arguments: Sequence[str]) -> None:
    ip = _require_ipython()
    command = " ".join(shlex.quote(str(arg)) for arg in arguments)
    print("pip", command)
    ip.run_line_magic("pip", command)


def pip_install(
    packages: Sequence[str],
    *,
    index_url: str | None = None,
    extra_index_url: str | None = None,
    upgrade: bool = True,
    extra_args: Sequence[str] | None = None,
) -> None:
    items = [pkg for pkg in packages if pkg]
    if not items:
        return
    args = ["install"]
    if upgrade:
        args.append("--upgrade")
    if index_url:
        args.extend(["--index-url", index_url])
    if extra_index_url:
        args.extend(["--extra-index-url", extra_index_url])
    if extra_args:
        args.extend(extra_args)
    args.extend(items)
    _run_pip(args)


def pip_uninstall(packages: Sequence[str]) -> None:
    items = [pkg for pkg in packages if pkg]
    if not items:
        return
    args = ["uninstall", "-y"]
    args.extend(items)
    _run_pip(args)


REPO_URL = "https://github.com/fouga1221/llm-lab2.git"
DEFAULT_REPO_DIR = "/content/llm-lab2" if "google.colab" in sys.modules else os.path.abspath("..")
REPO_DIR = DEFAULT_REPO_DIR
REPO_BRANCH = "main2"  # 必要に応じて変更

IS_COLAB = "google.colab" in sys.modules
DEFAULT_TORCH_INDEX = "https://download.pytorch.org/whl/cu126" if IS_COLAB else None
TORCH_INDEX_URL = os.environ.get("PYTORCH_WHL_INDEX_URL", DEFAULT_TORCH_INDEX)
TORCH_EXTRA_INDEX_URL = os.environ.get("PYTORCH_EXTRA_INDEX_URL", "https://pypi.org/simple")

BASE_BOOTSTRAP = [
    "pip>=24.2",
    "setuptools==79.0.1",
    "wheel>=0.44.0",
    "packaging>=24.2",
    "jedi>=0.19.1",
]

TORCH_PACKAGES = [
    "torch==2.8.0",
    "torchvision==0.23.0",
    "torchaudio==2.8.0",
]

TRAINING_CORE_PACKAGES = [
    "transformers==4.56.2",
    "peft==0.17.1",
    "accelerate==1.10.1",
    "bitsandbytes==0.44.1",
    "datasets==4.0.0",
    "evaluate==0.4.3",
    "trl==0.9.4",
    "safetensors>=0.4.2",
]

EXTRA_PACKAGES: Sequence[str] = []

clone_repo(REPO_URL, REPO_DIR, branch=REPO_BRANCH)
pip_install(BASE_BOOTSTRAP)
pip_install(TORCH_PACKAGES, index_url=TORCH_INDEX_URL, extra_index_url=TORCH_EXTRA_INDEX_URL)
pip_install(TRAINING_CORE_PACKAGES)
if EXTRA_PACKAGES:
    pip_install(EXTRA_PACKAGES)

print("依存パッケージのインストールが完了しました。ランタイム再起動が必要な場合は再起動後にこのセルと設定セルを再実行してください。")


In [ ]:
# === 2. パス設定とランタイム状態の永続化 ===
import json
import os
from pathlib import Path

IS_COLAB = "google.colab" in sys.modules

COLAB_DATA_ROOT_DEFAULT = Path("/content/drive/MyDrive/ProjectForte/llm-lab-save")
COLAB_WORKSPACE_ROOT_DEFAULT = Path("/content/llm-lab-save")
COLAB_DATA_ROOT = Path(os.environ.get("LLMLAB_DRIVE_DATA_ROOT", COLAB_DATA_ROOT_DEFAULT))
workspace_override = os.environ.get("LLMLAB_WORKSPACE_DATA_ROOT")
COLAB_WORKSPACE_ROOT = Path(workspace_override) if workspace_override else COLAB_WORKSPACE_ROOT_DEFAULT

if IS_COLAB:
    from google.colab import drive  # type: ignore

    skip_mount = os.environ.get("LLMLAB_SKIP_DRIVE_MOUNT", "").lower() in {"1", "true", "yes"}
    if skip_mount:
        print("環境変数 LLMLAB_SKIP_DRIVE_MOUNT により Google Drive マウントをスキップします。")
    else:
        drive.mount("/content/drive", force_remount=False)

    if workspace_override:
        COLAB_WORKSPACE_ROOT.mkdir(parents=True, exist_ok=True)
        DATA_ROOT = COLAB_WORKSPACE_ROOT
    else:
        COLAB_DATA_ROOT.mkdir(parents=True, exist_ok=True)
        if COLAB_WORKSPACE_ROOT.exists() or COLAB_WORKSPACE_ROOT.is_symlink():
            print("既存のワークスペースリンクを再利用します:", COLAB_WORKSPACE_ROOT)
        else:
            COLAB_WORKSPACE_ROOT.symlink_to(COLAB_DATA_ROOT, target_is_directory=True)
        DATA_ROOT = COLAB_WORKSPACE_ROOT
else:
    local_override = os.environ.get("LLMLAB_LOCAL_DATA_ROOT")
    DATA_ROOT = Path(local_override) if local_override else Path.cwd() / "save"
    DATA_ROOT.mkdir(parents=True, exist_ok=True)

REPO_ROOT = Path(REPO_DIR).resolve()
STATE_DIR = DATA_ROOT / "runtime_state"
STATE_DIR.mkdir(parents=True, exist_ok=True)
STATE_FILE = STATE_DIR / "finetuning_lora_qlora.json"

DEFAULT_CONFIG = {
    "BASE_MODEL_NAME": "Qwen/Qwen3-14B",
    "USE_QLORA": True,
    "BASE_MODEL_DTYPE": "bfloat16",
    "DATASET_PATH": str(DATA_ROOT / "datasets" / "train.jsonl"),
    "VALID_DATA_PATH": None,
    "OUTPUT_DIR": str(DATA_ROOT / "artifacts" / "lora_qwen3_14b"),
    "RUN_NAME": "qwen3_14b_lora_demo",
    "RESUME_FROM_CHECKPOINT": None,
    "TARGET_MODULES": [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    "LORA_R": 64,
    "LORA_ALPHA": 16,
    "LORA_DROPOUT": 0.05,
    "MERGE_TO_BASE": False,
    "MERGED_OUTPUT_DIR": str(DATA_ROOT / "artifacts" / "merged_qwen3_14b_ft"),
    "DATASET_FORMAT": {
        "TYPE": "prompt_response",
        "PROMPT_FIELD": "instruction",
        "INPUT_FIELD": "input",
        "RESPONSE_FIELD": "output",
        "TEMPLATE": (
            "### Instruction:\n{instruction}\n\n"
            "### Input:\n{input}\n\n"
            "### Response:\n{output}"
        ),
        "TEXT_FIELD": "text",
        "TRAIN_SPLIT": "train",
        "VALID_SPLIT": None,
    },
    "TRAINING_ARGS": {
        "output_dir": str(DATA_ROOT / "artifacts" / "lora_qwen3_14b" / "checkpoints"),
        "per_device_train_batch_size": 1,
        "gradient_accumulation_steps": 16,
        "num_train_epochs": 3,
        "learning_rate": 2e-4,
        "warmup_ratio": 0.03,
        "weight_decay": 0.0,
        "logging_steps": 10,
        "save_steps": 200,
        "save_total_limit": 3,
        "bf16": True,
        "fp16": False,
        "gradient_checkpointing": True,
        "report_to": ["tensorboard"],
    },
}


def _deep_merge(base: dict, override: dict) -> dict:
    result = dict(base)
    for key, value in override.items():
        if key in result and isinstance(result[key], dict) and isinstance(value, dict):
            result[key] = _deep_merge(result[key], value)
        else:
            result[key] = value
    return result


def load_state(path: Path, default: dict) -> dict:
    if not path.exists():
        return dict(default)
    try:
        payload = json.loads(path.read_text(encoding="utf-8"))
        print(f"保存済みの設定を復元しました: {path}")
        return _deep_merge(default, payload)
    except Exception as exc:
        print(f"設定ファイルの読み込みに失敗したためデフォルトを使用します: {exc}")
        return dict(default)


CONFIG = load_state(STATE_FILE, DEFAULT_CONFIG)
CONFIG["OUTPUT_DIR"] = str(Path(CONFIG["OUTPUT_DIR"]))
CONFIG["MERGED_OUTPUT_DIR"] = str(Path(CONFIG["MERGED_OUTPUT_DIR"]))
training_args_cfg = _deep_merge(DEFAULT_CONFIG["TRAINING_ARGS"], CONFIG.get("TRAINING_ARGS", {}))
if not training_args_cfg.get("output_dir"):
    training_args_cfg["output_dir"] = str(Path(CONFIG["OUTPUT_DIR"]) / "checkpoints")
CONFIG["TRAINING_ARGS"] = training_args_cfg


def persist_config(config: dict) -> None:
    STATE_FILE.write_text(json.dumps(config, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"設定を保存しました: {STATE_FILE}")


Path(CONFIG["OUTPUT_DIR"]).mkdir(parents=True, exist_ok=True)
Path(CONFIG["MERGED_OUTPUT_DIR"]).mkdir(parents=True, exist_ok=True)
Path(CONFIG["TRAINING_ARGS"]["output_dir"]).mkdir(parents=True, exist_ok=True)

persist_config(CONFIG)

print(f"REPO_ROOT: {REPO_ROOT}")
print(f"DATA_ROOT: {DATA_ROOT}")
print(f"OUTPUT_DIR: {CONFIG['OUTPUT_DIR']}")
print("設定を変更したら CONFIG を編集し、persist_config(CONFIG) を実行して保存してください。")


In [ ]:
# === 3. データセット読み込み ===
from pathlib import Path
from typing import Any, Optional

from datasets import Dataset, DatasetDict, load_dataset


def _load_local_dataset(path: Path) -> Dataset | DatasetDict:
    if path.is_dir():
        jsonl_files = sorted(path.glob("*.jsonl"))
        json_files = sorted(path.glob("*.json"))
        parquet_files = sorted(path.glob("*.parquet"))
        if jsonl_files:
            data_files = {"train": [str(p) for p in jsonl_files]}
            return load_dataset("json", data_files=data_files)
        if json_files:
            data_files = {"train": [str(p) for p in json_files]}
            return load_dataset("json", data_files=data_files)
        if parquet_files:
            data_files = {"train": [str(p) for p in parquet_files]}
            return load_dataset("parquet", data_files=data_files)
        raise FileNotFoundError(f"{path} に JSON/Parquet ファイルが見つかりません。")
    suffix = path.suffix.lower()
    if suffix in {".jsonl", ".json"}:
        return load_dataset("json", data_files={"train": str(path)})
    if suffix in {".parquet"}:
        return load_dataset("parquet", data_files={"train": str(path)})
    raise ValueError(f"対応していない拡張子です: {suffix}")


def _select_split(dataset: Dataset | DatasetDict, split: Optional[str]) -> Dataset:
    if isinstance(dataset, DatasetDict):
        target_split = split or "train"
        if target_split not in dataset:
            raise KeyError(f"指定された split {target_split} が見つかりません。利用可能: {list(dataset.keys())}")
        return dataset[target_split]
    return dataset


def _format_example_factory(fmt: dict[str, Any]):
    text_field = fmt.get("TEXT_FIELD", "text")
    fmt_type = fmt.get("TYPE", "prompt_response")
    prompt_field = fmt.get("PROMPT_FIELD", "instruction")
    input_field = fmt.get("INPUT_FIELD", "input")
    response_field = fmt.get("RESPONSE_FIELD", "output")
    template = fmt.get("TEMPLATE") or "{instruction}
{output}"

    def _formatter(example: dict[str, Any]) -> dict[str, str]:
        if fmt_type == "text":
            value = example.get(text_field)
            if value is None:
                raise KeyError(f"データに {text_field} フィールドがありません。")
            return {"text": value}
        instruction = example.get(prompt_field, "")
        input_text = example.get(input_field, "")
        output = example.get(response_field)
        if output is None:
            raise KeyError(f"データに {response_field} フィールドがありません。")
        text = template.format(
            instruction=instruction,
            input=input_text,
            output=output,
            **example,
        )
        return {"text": text.strip()}

    return _formatter


def load_training_corpus(config: dict) -> tuple[Dataset, Optional[Dataset]]:
    dataset_path = config["DATASET_PATH"]
    fmt = config["DATASET_FORMAT"]
    local_path = Path(dataset_path)
    if local_path.exists():
        dataset = _load_local_dataset(local_path)
    else:
        dataset = load_dataset(dataset_path)

    if isinstance(dataset, Dataset):
        dataset = DatasetDict({"train": dataset})

    train_ds = _select_split(dataset, fmt.get("TRAIN_SPLIT"))
    eval_ds = None

    valid_path = config.get("VALID_DATA_PATH")
    if valid_path:
        valid_local = Path(valid_path)
        if valid_local.exists():
            loaded_valid = _load_local_dataset(valid_local)
        else:
            loaded_valid = load_dataset(valid_path)
        if isinstance(loaded_valid, DatasetDict):
            eval_ds = _select_split(loaded_valid, fmt.get("VALID_SPLIT") or "validation")
        else:
            eval_ds = loaded_valid
    else:
        valid_split = fmt.get("VALID_SPLIT")
        if valid_split and isinstance(dataset, DatasetDict) and valid_split in dataset:
            eval_ds = dataset[valid_split]

    formatter = _format_example_factory(fmt)
    train_processed = train_ds.map(
        formatter,
        remove_columns=train_ds.column_names,
        desc="formatting train dataset",
        load_from_cache_file=False,
    )
    eval_processed = None
    if eval_ds is not None:
        eval_processed = eval_ds.map(
            formatter,
            remove_columns=eval_ds.column_names,
            desc="formatting eval dataset",
            load_from_cache_file=False,
        )

    return train_processed, eval_processed


TRAIN_DATASET, EVAL_DATASET = load_training_corpus(CONFIG)

print(TRAIN_DATASET)
if len(TRAIN_DATASET) > 0:
    print("サンプル:", TRAIN_DATASET[0]["text"][:200])

if EVAL_DATASET is not None:
    print(EVAL_DATASET)
else:
    print("評価用データセットは設定されていません。")


In [ ]:
# === 4. LoRA / QLoRA ファインチューニング ===
import gc
from pathlib import Path

import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

if "TRAIN_DATASET" not in globals() or TRAIN_DATASET is None:
    raise RuntimeError("TRAIN_DATASET が初期化されていません。セル3を実行してください。")

tokenizer = AutoTokenizer.from_pretrained(CONFIG["BASE_MODEL_NAME"], trust_remote_code=True)
added_tokens = 0
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({"pad_token": "<pad>"})
    tokenizer.pad_token = "<pad>"
    added_tokens = 1
tokenizer.padding_side = "right"

use_qlora = bool(CONFIG.get("USE_QLORA", True))
dtype_str = str(CONFIG.get("BASE_MODEL_DTYPE", "bfloat16")).lower()
dtype_map = {
    "float16": torch.float16,
    "fp16": torch.float16,
    "half": torch.float16,
    "bfloat16": torch.bfloat16,
    "bf16": torch.bfloat16,
    "float32": torch.float32,
    "fp32": torch.float32,
}
base_dtype = dtype_map.get(dtype_str, torch.bfloat16)

model_kwargs = {"trust_remote_code": True}
if use_qlora:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )
    model_kwargs["quantization_config"] = bnb_config
    model_kwargs["device_map"] = "auto"
else:
    model_kwargs["torch_dtype"] = base_dtype
    model_kwargs["device_map"] = "auto"

model = AutoModelForCausalLM.from_pretrained(
    CONFIG["BASE_MODEL_NAME"],
    **model_kwargs,
)
if added_tokens:
    model.resize_token_embeddings(len(tokenizer))
model.config.use_cache = False

if use_qlora:
    model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=int(CONFIG.get("LORA_R", 64)),
    lora_alpha=int(CONFIG.get("LORA_ALPHA", 16)),
    lora_dropout=float(CONFIG.get("LORA_DROPOUT", 0.05)),
    target_modules=list(CONFIG.get("TARGET_MODULES") or []),
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

train_args_dict = dict(CONFIG["TRAINING_ARGS"])
train_args_dict.setdefault("output_dir", CONFIG["TRAINING_ARGS"]["output_dir"])
train_args_dict.setdefault("run_name", CONFIG.get("RUN_NAME", "finetune_run"))
training_args = TrainingArguments(**train_args_dict)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=TRAIN_DATASET,
    eval_dataset=EVAL_DATASET,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

resume_ckpt = CONFIG.get("RESUME_FROM_CHECKPOINT")
train_result = trainer.train(resume_from_checkpoint=resume_ckpt)
trainer.save_model(CONFIG["OUTPUT_DIR"])
tokenizer.save_pretrained(CONFIG["OUTPUT_DIR"])

CONFIG["LAST_METRICS"] = train_result.metrics
CONFIG["LAST_CHECKPOINT"] = trainer.state.best_model_checkpoint or training_args.output_dir
persist_config(CONFIG)

print("学習が完了しました。LoRA アダプタは次のパスに保存されています:", CONFIG["OUTPUT_DIR"])


In [ ]:
# === 5. (任意) LoRA アダプタをベースモデルへマージ ===
import gc
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

if not CONFIG.get("MERGE_TO_BASE", False):
    print("CONFIG['MERGE_TO_BASE'] が False のためマージは実行しません。")
else:
    output_dir = Path(CONFIG["MERGED_OUTPUT_DIR"])
    output_dir.mkdir(parents=True, exist_ok=True)
    print("CPU 上でベースモデルに LoRA をマージします...")
    base_model = AutoModelForCausalLM.from_pretrained(
        CONFIG["BASE_MODEL_NAME"],
        device_map={"": "cpu"},
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )
    lora_model = PeftModel.from_pretrained(base_model, CONFIG["OUTPUT_DIR"])
    merged = lora_model.merge_and_unload()
    merged.save_pretrained(output_dir, safe_tensors=True)
    tokenizer = AutoTokenizer.from_pretrained(CONFIG["BASE_MODEL_NAME"], trust_remote_code=True)
    tokenizer.save_pretrained(output_dir)
    del base_model, lora_model, merged, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    CONFIG["LAST_MERGED_DIR"] = str(output_dir)
    persist_config(CONFIG)
    print("マージ済みモデルを保存しました:", output_dir)


In [ ]:
# === 6. 後片付け ===
import gc
import torch

for name in ["model", "trainer"]:
    if name in globals():
        del globals()[name]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("GPU メモリを解放しました。必要に応じて次の作業に進んでください。")
